In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#!pip install -q --force-reinstall "numpy==1.26.4" "scikit-surprise==1.1.4"

#!pip install -q fastapi uvicorn nest_asyncio streamlit pyngrok joblib pandas==2.2.* scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 577.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.7/307.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 13.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Caricamento**

In [3]:
# Funzionalità 1 - Caricamento dei modelli e dei dati

import joblib
import numpy as np
import pandas as pd
import json

# Percorso della cartella dei modelli su Google Drive
LOAD_DIR = '/content/drive/MyDrive/TESI/models'

# Caricamento dei modelli
svd = joblib.load(f'{LOAD_DIR}/svd.pkl')                                # Modello di collaborative filtering (SVD)
cos_sim = np.load(f'{LOAD_DIR}/cosine_sim_bert.npy')                    # Matrice di similarità per content-based con BERT
films_df = pd.read_pickle(f'{LOAD_DIR}/final_movies_df.pkl')           # DataFrame con metadati completi dei film
id_map = pd.read_pickle(f'{LOAD_DIR}/id_map.pkl')                       # Mappatura interna tra movieId e indice della matrice
with open(f'{LOAD_DIR}/config.json') as f:
    alpha_default = json.load(f)["alpha_default"]                      # Valore predefinito di alpha (fusione ibrida)

# Output di controllo
print("Modelli e dati caricati correttamente!")
print(f"Numero di film disponibili: {films_df.shape[0]}")


Modelli e dati caricati correttamente!
Numero di film disponibili: 9219


In [4]:
# Calcolo score_bert per ciascun film (media della similarità con gli altri film)
cos_sim_no_diag = cos_sim.copy()
np.fill_diagonal(cos_sim_no_diag, np.nan)  # Escludi self-similarity (film con se stesso)

films_df['score_bert'] = np.nanmean(cos_sim_no_diag, axis=1)

# Salvataggio del nuovo films_df con score_bert incluso
films_df.to_pickle(f'{LOAD_DIR}/final_movies_df_with_score.pkl')

print("Calcolato e aggiunto 'score_bert' a films_df.")


Calcolato e aggiunto 'score_bert' a films_df.


In [5]:
ratings = pd.read_csv('/content/drive/MyDrive/TESI/ratings_small.csv')
links = pd.read_csv('/content/drive/MyDrive/TESI/links_small.csv')
movies_metadata = pd.read_csv('/content/drive/MyDrive/TESI/movies_metadata.csv', low_memory=False)

links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
movies_metadata['id'] = pd.to_numeric(movies_metadata['id'], errors='coerce')
movies_metadata['release_date'] = pd.to_datetime(movies_metadata['release_date'], errors='coerce')
movies_metadata['year'] = movies_metadata['release_date'].dt.year
movies_metadata['runtime'] = pd.to_numeric(movies_metadata['runtime'], errors='coerce')

# Questo è il vero id_map utile per unire SVD e metadati
id_map = links.merge(movies_metadata[['id', 'title']], left_on='tmdbId', right_on='id')
id_map = id_map[['movieId', 'id', 'title']]

# (Opzionale) aggiorniamo anche ratings con il titolo
ratings = ratings.merge(id_map[['movieId', 'title']], on='movieId', how='left')

filtered_df = films_df.copy()
filtered_df = filtered_df.merge(movies_metadata[['id', 'year', 'runtime']], on='id', how='left')


**Scelta utente (Collaborative Fil)**

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# Caricamento dei ratings
ratings = pd.read_csv('/content/drive/MyDrive/TESI/ratings_small.csv')

# Prendiamo solo utenti con almeno 20 rating e li ordiniamo in modo crescente
valid_user_ids = (
    ratings['userId'].value_counts()
    .loc[lambda x: x >= 20]
    .index.sort_values()
    .tolist()
)

# ✅ Variabile globale per la selezione utente
selected_user_id = None

# ✅ UI per la selezione utente (SVD)
def get_user_id_ui():
    global selected_user_id

    # Menu a tendina per scegliere l'utente
    user_selector = widgets.Dropdown(
        options=valid_user_ids,
        description='User ID:',
        layout=widgets.Layout(width='300px')
    )

    # Bottone per confermare la scelta
    confirm_button = widgets.Button(
        description="🎯 Avvia la raccomandazione",
        button_style='success',
        layout=widgets.Layout(width='300px')
    )

    # NON stampiamo nulla subito
    selected_user_id = None

    # Aggiorna la variabile solo quando l'utente cambia
    def on_user_selected(change):
        global selected_user_id
        if change['type'] == 'change' and change['name'] == 'value':
            selected_user_id = change['new']
            print(f"🔑 Hai selezionato l'utente: {selected_user_id}")

    # Se nessuna scelta è fatta, usa il valore corrente della tendina
    def on_confirm_clicked(b):
        global selected_user_id
        if selected_user_id is None:
            selected_user_id = user_selector.value
        clear_output()
        get_top10_svd()

    user_selector.observe(on_user_selected)
    confirm_button.on_click(on_confirm_clicked)

    display(user_selector, confirm_button)


**Scelta utente (hybrid)**

In [7]:
# ✅ UI per la selezione utente (Funzionalità 2 – Ibrido)
def get_user_id_ui_hybrid():
    global selected_user_id

    # Menu a tendina per scegliere l'utente
    user_selector = widgets.Dropdown(
        options=valid_user_ids,
        description='User ID:',
        layout=widgets.Layout(width='300px')
    )

    # Bottone per confermare la scelta
    confirm_button = widgets.Button(
        description="🔀 Avvia la raccomandazione ibrida",
        button_style='success',
        layout=widgets.Layout(width='300px')
    )

    # NON stampiamo nulla subito
    selected_user_id = None

    # Aggiorna la variabile solo se l'utente cambia selezione
    def on_user_selected(change):
        global selected_user_id
        if change['type'] == 'change' and change['name'] == 'value':
            selected_user_id = change['new']
            print(f"🔑 Hai selezionato l'utente: {selected_user_id}")

    # Se nessuna selezione manuale, usa quella corrente della tendina
    def on_confirm_clicked(b):
        global selected_user_id
        if selected_user_id is None:
            selected_user_id = user_selector.value
        clear_output()
        get_top10_hybrid()

    user_selector.observe(on_user_selected)
    confirm_button.on_click(on_confirm_clicked)

    display(user_selector, confirm_button)



**Scelta film (content)**

In [8]:
def get_similar_movies_ui():
    search_input = widgets.Text(
        placeholder='Scrivi parte del titolo del film...',
        description='Cerca:',
        layout=widgets.Layout(width='500px')
    )

    filtered_selector = widgets.Dropdown(
        options=[],
        description='Scegli:',
        layout=widgets.Layout(width='500px')
    )

    confirm_button = widgets.Button(
        description="🎬 Mostra film simili",
        button_style='success',
        layout=widgets.Layout(width='300px')
    )

    output_box = widgets.Output()

    def on_search_change(change):
        with output_box:
            output_box.clear_output()
            query = search_input.value.strip().lower()
            if query == "":
                filtered_selector.options = []
                return
            matches = films_df['title'][films_df['title'].str.lower().str.contains(query, na=False)].sort_values().unique().tolist()
            if matches:
                filtered_selector.options = matches
            else:
                filtered_selector.options = []
                print("❌ Nessun titolo trovato.")

    def on_confirm_clicked(b):
        selected_title = filtered_selector.value
        if selected_title:
            clear_output()
            get_similar_movies(selected_title)
        else:
            with output_box:
                output_box.clear_output()
                print("⚠️ Seleziona un titolo valido prima di procedere.")

    search_input.observe(on_search_change, names='value')
    confirm_button.on_click(on_confirm_clicked)

    display(widgets.VBox([search_input, filtered_selector, confirm_button, output_box]))




**Filtro genere anno durata**

In [9]:
def get_filtered_movies_ui():
    from IPython.display import display

    # Funzione robusta per estrarre generi
    def extract_genres(g):
        if isinstance(g, str):
            try:
                g = eval(g)
            except:
                return []
        return g if isinstance(g, list) else []

    all_genres = sorted({
        genre.strip()
        for g in filtered_df['genres'].dropna()
        for genre in extract_genres(g)
        if isinstance(genre, str)
    })

    # ✅ Lista di checkbox
    genre_checkboxes = [widgets.Checkbox(value=False, description=genre) for genre in all_genres]

    genre_box = widgets.VBox(genre_checkboxes, layout=widgets.Layout(max_height='200px', overflow='auto'))

    # Slider per anno
    year_min = int(filtered_df['year'].dropna().min())
    year_max = int(filtered_df['year'].dropna().max())

    year_slider = widgets.IntRangeSlider(
        value=[year_min, year_max],
        min=year_min,
        max=year_max,
        step=1,
        description='Anno:',
        layout=widgets.Layout(width='400px')
    )

    # Slider per durata
    runtime_min = int(filtered_df['runtime'].dropna().min())
    runtime_max = int(filtered_df['runtime'].dropna().max())

    runtime_slider = widgets.IntRangeSlider(
        value=[runtime_min, runtime_max],
        min=runtime_min,
        max=runtime_max,
        step=5,
        description='Durata:',
        layout=widgets.Layout(width='400px')
    )

    confirm_button = widgets.Button(
        description="🎛️ Applica filtri",
        button_style='success',
        layout=widgets.Layout(width='300px')
    )

    output_box = widgets.Output()

    def on_confirm_clicked(b):
        spinner_output.clear_output()
        output_box.clear_output()
        display(spinner_output)
        with spinner_output:
            display(spinner_html)

        # Leggi i generi selezionati
        selected_genres = {cb.description for cb in genre_checkboxes if cb.value}

        year_range = year_slider.value
        runtime_range = runtime_slider.value

        filtered = filtered_df.dropna(subset=['genres', 'year', 'runtime']).copy()

        # Filtro anno + durata
        filtered = filtered[
            (filtered['year'] >= year_range[0]) & (filtered['year'] <= year_range[1]) &
            (filtered['runtime'] >= runtime_range[0]) & (filtered['runtime'] <= runtime_range[1])
        ]

        # Filtro per generi (tutti i selezionati devono essere contenuti)
        if selected_genres:
            def has_genres(g):
                genres = extract_genres(g)
                return selected_genres.issubset(set(genres)) if genres else False
            filtered = filtered[filtered['genres'].apply(has_genres)]

        spinner_output.clear_output()
        if filtered.empty:
            with output_box:
                print("❌ Nessun film trovato con i criteri selezionati.")
        else:
            stampa_film_cards(filtered.sort_values('year', ascending=False).head(10))


        mostra_opzioni_post_risultato(get_filtered_movies_ui, "🔁 Cambia i filtri")

    confirm_button.on_click(on_confirm_clicked)

    display(widgets.VBox([
        widgets.Label("🎯 Seleziona uno o più generi:"),
        genre_box,
        year_slider,
        runtime_slider,
        confirm_button,
        output_box
    ]))



**Get gemme nascoste**

In [10]:
def get_gemme_nascoste_ui():
    button = widgets.Button(
        description="💎 Scopri ora",
        button_style='success',
        layout=widgets.Layout(width='250px')
    )

    output_box = widgets.Output()

    def on_click(b):
        spinner_output.clear_output()
        output_box.clear_output()
        display(spinner_output)
        with spinner_output:
            display(spinner_html)

        # 📊 Statistiche da ratings: numero e media dei voti per ogni movieId
        rating_stats = ratings.groupby('movieId')['rating'].agg(
            valutato_da_utenti='count',
            media_valutazioni='mean'
        ).reset_index()

        # 🎯 Seleziona film poco popolari (< 20 rating)
        pochi_rating_ids = rating_stats[rating_stats['valutato_da_utenti'] < 20]['movieId'].tolist()
        gems = id_map[id_map['movieId'].isin(pochi_rating_ids)]

        # Unione con score_bert e metadati
        df = gems.merge(films_df[['id', 'title', 'genres', 'score_bert']], on='id', how='left')
        if 'year' in films_df.columns and 'runtime' in films_df.columns:
            df = df.merge(films_df[['id', 'year', 'runtime']], on='id', how='left')

        # Aggiunta delle statistiche dai ratings
        df = df.merge(rating_stats, on='movieId', how='left')

        # Fix dei titoli
        df.drop(columns=['title_x'], inplace=True)
        df.rename(columns={'title_y': 'title'}, inplace=True)

        # Pulizia e ordinamento
        df = df.dropna(subset=['score_bert', 'genres'])
        df = df.sort_values('score_bert', ascending=False).head(10)

        spinner_output.clear_output()
        if df.empty:
            with output_box:
                print("❌ Nessuna gemma nascosta trovata.")
        else:
            stampa_film_cards(df)

        mostra_opzioni_post_risultato(get_gemme_nascoste_ui, "🔁 Cerca altre gemme")

    button.on_click(on_click)
    display(widgets.VBox([button, output_box]))


**Filtro negativo**

In [11]:
# Variabile globale per il filtro negativo
filtro_negativo_titolo = None

**Spinner caricamento**

In [12]:
spinner_output = widgets.Output()

spinner_html = widgets.HTML(
    """
    <div style="display: flex; align-items: center; gap: 12px;">
        <div style="font-size: 24px; animation: rotate 1s linear infinite;">⏳</div>
        <b>Caricamento in corso...</b>
    </div>
    <style>
    @keyframes rotate {
        0%   { transform: rotate(0deg); }
        100% { transform: rotate(360deg); }
    }
    </style>
    """
)


**Output HTML**

In [13]:
from IPython.display import display, HTML

def stampa_film_cards(df):
    html = "<div style='font-family: sans-serif;'>"

    # Colonne dinamiche
    has_year = 'year' in df.columns
    has_runtime = 'runtime' in df.columns
    has_rating = 'predicted_rating' in df.columns
    has_score = 'score_hybrid' in df.columns
    has_sim = 'similarity_score' in df.columns
    has_nutenti = 'valutato_da_utenti' in df.columns
    has_avg = 'media_valutazioni' in df.columns
    has_spiegazione = 'spiegazione' in df.columns

    # Conta colonne per colspan
    n_col = 2  # titolo, generi
    for c in [has_year, has_runtime, has_rating, has_score, has_sim, has_nutenti, has_avg]:
        if c: n_col += 1

    # Header dinamico
    html += """
    <style>
        .film-table {
            font-family: sans-serif;
            border-collapse: collapse;
            width: 100%;
        }
        .film-table th {
            background-color: #f2f2f2;
            padding: 8px;
            text-align: left;
            border-bottom: 2px solid #ddd;
        }
        .film-table td {
            padding: 8px;
            border-bottom: 1px solid #ddd;
        }
        .film-table tr:nth-child(even) {
            background-color: #f9f9f9;
        }
        .spiegazione-row {
            color: #33506a;
            font-size: 13px;
            background: #f3f6fa;
        }
    </style>
    <table class="film-table">
        <tr>
            <th>🎬 Titolo</th>
            <th>🎭 Generi</th>"""
    if has_year:
        html += "<th>📆 Anno</th>"
    if has_runtime:
        html += "<th>⏱️ Durata</th>"
    if has_rating:
        html += "<th>⭐ Punteggio stimato</th>"
    if has_score:
        html += "<th>⚖️ Score ibrido</th>"
    if has_sim:
        html += "<th>🔗 Similarità</th>"
    if has_nutenti:
        html += "<th>🧑‍🤝‍🧑 Valutato da</th>"
    if has_avg:
        html += "<th>⭐ Media voti</th>"
    html += "</tr>"

    # Righe della tabella
    for _, row in df.iterrows():
        title = row.get("title", "—")

        g = row.get("genres")
        if isinstance(g, str):
            try:
                g = eval(g)
            except:
                g = []
        genres = ', '.join(g) if isinstance(g, list) else "—"

        year = int(row["year"]) if has_year and not pd.isna(row.get("year")) else "—"
        runtime = int(row["runtime"]) if has_runtime and not pd.isna(row.get("runtime")) else "—"
        rating = f"{row['predicted_rating']:.2f}" if has_rating else ""
        score = f"{row['score_hybrid']:.2f}" if has_score else ""
        sim = f"{row['similarity_score']:.2f}" if has_sim else ""
        utenti = f"{int(row['valutato_da_utenti'])}" if has_nutenti else ""
        avg = f"{row['media_valutazioni']:.2f}" if has_avg else ""

        html += f"""
        <tr>
            <td><b>{title}</b></td>
            <td>{genres}</td>"""
        if has_year:
            html += f"<td>{year}</td>"
        if has_runtime:
            html += f"<td>{runtime} min</td>"
        if has_rating:
            html += f"<td>{rating}</td>"
        if has_score:
            html += f"<td>{score}</td>"
        if has_sim:
            html += f"<td>{sim}</td>"
        if has_nutenti:
            html += f"<td>{utenti} utenti</td>"
        if has_avg:
            html += f"<td>{avg}</td>"
        html += "</tr>"

        # Spiegazione sotto
        if has_spiegazione:
            spiegazione = row['spiegazione']
            html += f"<tr><td class='spiegazione-row' colspan='{n_col}'>💬 <i>{spiegazione}</i></td></tr>"

    html += "</table></div>"
    display(HTML(html))


**Riprova o menù principale**

In [14]:
def mostra_opzioni_post_risultato(riprova_callback, riprova_label, mostra_filtro_negativo=False, last_df=None, selected_title=None):
    from IPython.display import display, clear_output
    import ipywidgets as widgets

    bottone_riprova = widgets.Button(
        description=f"🔁 {riprova_label}",
        button_style='info',
        layout=widgets.Layout(width='250px')
    )

    bottone_menu = widgets.Button(
        description="🏠 Torna al menù principale",
        button_style='warning',
        layout=widgets.Layout(width='250px')
    )

    buttons = [bottone_riprova, bottone_menu]

    # Nuovo bottone filtro negativo
    if mostra_filtro_negativo and last_df is not None:
        bottone_filtro = widgets.Button(
            description="🚫 Escludi film simili a un titolo",
            button_style='danger',
            layout=widgets.Layout(width='320px')
        )

        def on_filtro_clicked(b):
            clear_output()
            mostra_filtro_negativo_ui(riprova_callback, last_df, selected_title)

        bottone_filtro.on_click(on_filtro_clicked)
        buttons.append(bottone_filtro)

    def on_riprova_clicked(b):
        global selected_user_id
        selected_user_id = None  # Reset (fondamentale)
        clear_output()
        riprova_callback()

    def on_menu_clicked(b):
        global selected_user_id
        selected_user_id = None
        clear_output()
        display(function_selector, run_button)

    bottone_riprova.on_click(on_riprova_clicked)
    bottone_menu.on_click(on_menu_clicked)

    display(widgets.HBox(buttons))



**Filtro negativo**

In [15]:
def mostra_filtro_negativo_ui(rerun_func, last_df, selected_title=None):
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    titoli = sorted(last_df['title'].dropna().unique())
    titolo_input = widgets.Combobox(
        options=titoli,
        placeholder='Digita per cercare...',
        description='Titolo:',
        ensure_option=True,
        layout=widgets.Layout(width='500px')
    )
    bottone_applica = widgets.Button(
        description="🚫 Escludi simili e ricalcola",
        button_style='danger',
        layout=widgets.Layout(width='320px')
    )

    def on_apply(b):
        global filtro_negativo_titolo
        filtro_negativo_titolo = titolo_input.value if titolo_input.value else None
        clear_output()
        if selected_title:
            rerun_func(selected_title)
        else:
            rerun_func()

    bottone_applica.on_click(on_apply)
    display(widgets.VBox([titolo_input, bottone_applica]))



**Cold start**

In [16]:
def get_coldstart_fallback(user_id):
    spinner_output.clear_output()
    display(spinner_output)
    with spinner_output:
        display(spinner_html)

    # Messaggio user-friendly
    print("👋 È la tua prima volta qui? Nessun problema!")
    print("🎯 Ti consigliamo alcuni film basandoci sulle loro caratteristiche, finché non avremo più informazioni sui tuoi gusti.\n")

    # Escludi i film già visti
    seen = ratings[ratings['userId'] == user_id]['movieId'].tolist()
    unseen_df = id_map[~id_map['movieId'].isin(seen)]

    # Unione con films_df + score_bert
    merged = unseen_df.merge(films_df[['id', 'title', 'genres', 'score_bert']], on='id', how='left')

    if 'year' in films_df.columns and 'runtime' in films_df.columns:
        merged = merged.merge(films_df[['id', 'year', 'runtime']], on='id', how='left')

    merged = merged.dropna(subset=['score_bert', 'genres'])
    merged = merged.sort_values('score_bert', ascending=False).head(10)

    if merged.empty:
        spinner_output.clear_output()
        print("⚠️ Nessuna alternativa trovata.")
        return

    spinner_output.clear_output()
    stampa_film_cards(merged)

    mostra_opzioni_post_risultato(get_user_id_ui, "🎭 Prova con un altro profilo")


**Top 10 personalizzati (SDV)**

In [17]:
def get_top10_svd():
    global selected_user_id, filtro_negativo_titolo

    if selected_user_id is None:
        print("⚠️ Questa funzione richiede la selezione di un utente.")
        get_user_id_ui()
        return

    # Cold-start check
    user_ratings = ratings[ratings['userId'] == selected_user_id]
    if user_ratings.shape[0] < 5:
        print("🧊 Utente con pochi rating: fallback a raccomandazioni content-based.")
        get_coldstart_fallback(selected_user_id)
        return

    spinner_output.clear_output()
    display(spinner_output)
    with spinner_output:
        display(spinner_html)

    all_movie_ids = id_map['movieId'].dropna().unique().tolist()
    seen_movies = user_ratings['movieId'].tolist()
    unseen_movies = [m for m in all_movie_ids if m not in seen_movies]

    predictions = []
    for movie_id in unseen_movies:
        pred = svd.predict(selected_user_id, movie_id)
        if pred.details['was_impossible']:
            continue
        predictions.append((movie_id, pred.est))

    if not predictions:
        spinner_output.clear_output()
        print("⚠️ Nessuna raccomandazione disponibile per questo utente. Prova con un altro profilo.")
        mostra_opzioni_post_risultato(get_top10_svd, "Prova con un altro profilo")
        return

    top10_df = pd.DataFrame(sorted(predictions, key=lambda x: x[1], reverse=True),
                            columns=['movieId', 'predicted_rating'])

    top10_df = top10_df.merge(id_map[['movieId', 'id', 'title']], on='movieId', how='left')
    top10_df = top10_df.merge(films_df[['id', 'title', 'genres']], on='id', how='left')

    if 'year' in films_df.columns and 'runtime' in films_df.columns:
        top10_df = top10_df.merge(films_df[['id', 'year', 'runtime']], on='id', how='left')

    top10_df.dropna(subset=['title_y', 'genres'], inplace=True)
    top10_df.drop(columns=['title_x'], inplace=True)
    top10_df.rename(columns={'title_y': 'title'}, inplace=True)

    # Applica filtro negativo se attivo
    if filtro_negativo_titolo:
        try:
            idx = films_df[films_df['title'] == filtro_negativo_titolo].index[0]
            sim_scores = list(enumerate(cos_sim[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            exclude_indices = [s[0] for s in sim_scores[:20]] + [idx]
            exclude_ids = films_df.iloc[exclude_indices]['id'].tolist()
            top10_df = top10_df[~top10_df['id'].isin(exclude_ids)]
        except Exception:
            pass
        filtro_negativo_titolo = None  # Reset dopo l'uso

    # Mostra i primi 10
    top10_df = top10_df.head(10)

    spinner_output.clear_output()
    stampa_film_cards(top10_df)

    mostra_opzioni_post_risultato(get_top10_svd, "🎭 Prova con un altro profilo", mostra_filtro_negativo=True, last_df=top10_df)




**Raccomandazione ibrida (score-based)** 🎭

In [18]:
def get_top10_hybrid():
    global selected_user_id, filtro_negativo_titolo

    if selected_user_id is None:
        print("⚠️ Questa funzione richiede la selezione di un utente.")
        get_user_id_ui_hybrid()
        return

    # Cold-start check
    user_ratings = ratings[ratings['userId'] == selected_user_id]
    if user_ratings.shape[0] < 5:
        get_coldstart_fallback(selected_user_id)
        return

    spinner_output.clear_output()
    display(spinner_output)
    with spinner_output:
        display(spinner_html)

    all_movie_ids = id_map['movieId'].dropna().unique().tolist()
    seen = user_ratings['movieId'].tolist()
    unseen = [m for m in all_movie_ids if m not in seen]

    results = []
    for movie_id in unseen:
        row = id_map[id_map['movieId'] == movie_id]
        if row.empty:
            continue
        film_id = row['id'].values[0]
        if film_id not in films_df['id'].values:
            continue
        try:
            score_bert = films_df.loc[films_df['id'] == film_id, 'score_bert'].values[0]
        except IndexError:
            continue
        if pd.isna(score_bert):
            continue

        pred = svd.predict(selected_user_id, movie_id)
        if pred.details['was_impossible']:
            continue
        score_svd = pred.est
        score = alpha_default * score_bert + (1 - alpha_default) * score_svd
        results.append((movie_id, score))

    if not results:
        spinner_output.clear_output()
        print("⚠️ Nessuna raccomandazione disponibile per questo utente. Prova con un altro profilo.")
        mostra_opzioni_post_risultato(get_top10_hybrid, "Prova con un altro profilo")
        return

    top10_df = pd.DataFrame(sorted(results, key=lambda x: x[1], reverse=True),
                            columns=['movieId', 'score_hybrid'])

    top10_df = top10_df.merge(id_map[['movieId', 'id', 'title']], on='movieId', how='left')
    top10_df = top10_df.merge(films_df[['id', 'title', 'genres']], on='id', how='left')

    if 'year' in films_df.columns and 'runtime' in films_df.columns:
        top10_df = top10_df.merge(films_df[['id', 'year', 'runtime']], on='id', how='left')

    top10_df.dropna(subset=['title_y', 'genres'], inplace=True)
    top10_df.drop(columns=['title_x'], inplace=True)
    top10_df.rename(columns={'title_y': 'title'}, inplace=True)

    # Applica filtro negativo se attivo
    if filtro_negativo_titolo:
        try:
            idx = films_df[films_df['title'] == filtro_negativo_titolo].index[0]
            sim_scores = list(enumerate(cos_sim[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            exclude_indices = [s[0] for s in sim_scores[:20]] + [idx]
            exclude_ids = films_df.iloc[exclude_indices]['id'].tolist()
            top10_df = top10_df[~top10_df['id'].isin(exclude_ids)]
        except Exception:
            pass
        filtro_negativo_titolo = None  # Reset dopo l'uso

    # Mostra i primi 10
    top10_df = top10_df.head(10)

    spinner_output.clear_output()
    stampa_film_cards(top10_df)

    mostra_opzioni_post_risultato(get_top10_hybrid, "🎭 Prova con un altro profilo", mostra_filtro_negativo=True, last_df=top10_df)



**Raccomandazione simili (content-based)**

In [19]:
def get_similar_movies(title):
    spinner_output.clear_output()
    display(spinner_output)
    with spinner_output:
        display(spinner_html)

    try:
        film_index = films_df[films_df['title'] == title].index[0]
    except IndexError:
        spinner_output.clear_output()
        print("❌ Errore: titolo non trovato.")
        return

    sim_scores = list(enumerate(cos_sim[film_index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != film_index]
    top10 = sim_scores[:10]

    indices_top10 = [i[0] for i in top10]
    score_top10 = [i[1] for i in top10]

    result_df = films_df.loc[indices_top10, ['id', 'title', 'genres']].copy()
    result_df['similarity_score'] = score_top10

    if 'year' in films_df.columns and 'runtime' in films_df.columns:
        result_df = result_df.merge(films_df[['id', 'year', 'runtime']], on='id', how='left')

    spinner_output.clear_output()
    stampa_film_cards(result_df)

    mostra_opzioni_post_risultato(get_similar_movies_ui, "🎞️ Scegli un altro film")




**Interfaccia**

In [20]:
from IPython.display import display, clear_output
import ipywidgets as widgets

# Etichette visibili al cliente
function_options = [
    '📋 Scegli cosa vuoi fare...',
    '🎯 Ricevi 10 consigli personalizzati in base ai tuoi gusti',
    '🔀 Scopri film perfetti per te combinando gusti e contenuti',
    '🎬 Hai amato un film? Ti mostriamo altri simili',
    '🎛️ Filtra i consigli per genere, anno di uscita o durata',
    '💎 Scopri gemme nascoste: film poco noti ma con ottime valutazioni',
]

# Dropdown e bottone
function_selector = widgets.Dropdown(
    options=function_options,
    description='Funzionalità:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

run_button = widgets.Button(description="Avvia", button_style='success')

# Azione del bottone
def on_run_button_clicked(b):
    clear_output()
    display(function_selector, run_button)

    scelta = function_selector.value

    if scelta == '🎯 Ricevi 10 consigli personalizzati in base ai tuoi gusti':
        get_top10_svd()
    elif scelta == '🔀 Scopri film perfetti per te combinando gusti e contenuti':
        get_top10_hybrid()
    elif scelta == '🎬 Hai amato un film? Ti mostriamo altri simili':
        get_similar_movies_ui()
    elif scelta == '🎛️ Filtra i consigli per genere, anno di uscita o durata':
        get_filtered_movies_ui()
    elif scelta == '💎 Scopri gemme nascoste: film poco noti ma con ottime valutazioni':
        get_gemme_nascoste_ui()
    else:
        print("⚠️ Funzionalità non ancora attiva. A breve sarà disponibile.")

run_button.on_click(on_run_button_clicked)

# Visualizza il menu
display(function_selector, run_button)


Dropdown(description='Funzionalità:', index=5, layout=Layout(width='80%'), options=('📋 Scegli cosa vuoi fare..…

Button(button_style='success', description='Avvia', style=ButtonStyle())